# Задача 4.17 — Треугольник из трёх точек в кубе (Монте-Карло)

---

## 1. Условие

В кубе объёма 1 случайным образом выбираются точки $A, B, C$. События:

- $R$ — наименьший угол в треугольнике $ABC$ меньше $26{,}7^\circ$;
- $S$ — наибольший угол в треугольнике $ABC$ меньше $81{,}9^\circ$;
- $T$ — треугольник $ABC$ остроугольный.

Найдите приближённо методом Монте-Карло:

1) $P(R \mid T)$;

2) $P(S \mid T)$.

Указание: получить 100 000 остроугольных треугольников, ответы округлить до одного знака после запятой.

**Ответ:** 1) $P(R\mid T) = 0{,}11$; 2) $P(S\mid T) = 0{,}65$.

---

## 2. Теоретическая часть

Генерируем три случайные точки в единичном кубе. Вычисляем длины сторон и углы треугольника по теореме косинусов. Остроугольность — все три угла $< 90^\circ$. Для остроугольных треугольников проверяем условия на минимальный и максимальный углы.

---

## 3. Решение

In [1]:
import numpy as np

N_target = 100_000
rng = np.random.default_rng(seed=16)

def angles_of_triangle(P1, P2, P3):
    """Возвращает углы (в градусах) треугольника с вершинами P1, P2, P3."""
    a = np.linalg.norm(P2 - P3, axis=1)
    b = np.linalg.norm(P1 - P3, axis=1)
    c = np.linalg.norm(P1 - P2, axis=1)
    # Теорема косинусов
    cosA = (b**2 + c**2 - a**2) / (2*b*c)
    cosB = (a**2 + c**2 - b**2) / (2*a*c)
    cosC = (a**2 + b**2 - c**2) / (2*a*b)
    cosA = np.clip(cosA, -1, 1); cosB = np.clip(cosB, -1, 1); cosC = np.clip(cosC, -1, 1)
    return np.degrees(np.arccos(cosA)), np.degrees(np.arccos(cosB)), np.degrees(np.arccos(cosC))

# Накапливаем остроугольные треугольники
acute_angles = []
batch = 100_000
while sum(len(x) for x in acute_angles) < N_target:
    P1 = rng.random((batch, 3))
    P2 = rng.random((batch, 3))
    P3 = rng.random((batch, 3))
    A, B, C = angles_of_triangle(P1, P2, P3)
    is_acute = (A < 90) & (B < 90) & (C < 90)
    if is_acute.any():
        acute_angles.append(np.column_stack([A[is_acute], B[is_acute], C[is_acute]]))

angles = np.vstack(acute_angles)[:N_target]
min_ang = angles.min(axis=1)
max_ang = angles.max(axis=1)

R = min_ang < 26.7
S = max_ang < 81.9

print(f"1) P(R|T) ≈ {R.mean():.2f}  (ответ: 0.11)")
print(f"2) P(S|T) ≈ {S.mean():.2f}  (ответ: 0.65)")

1) P(R|T) ≈ 0.11  (ответ: 0.11)
2) P(S|T) ≈ 0.65  (ответ: 0.65)
